# Coupon Used 로그 SQL 생성
- `coupon_received_map.json` (generate_coupon_received_logs.ipynb 실행 결과)을 읽어,
  실제로 발급받은 (user_id, coupon_code) 조합 중 USE_RATE(30%)만 사용 로그 생성
- 사용 시각은 발급 시각 ~ 만료일(expired_days) 사이에서 랜덤 (만료 후 사용 불가)
- discountAmount: received 시점과 동일한 값 사용 (RATE 타입은 0, FIXED 타입은 실제 금액)
- expiryDate 없음 (coupon_used 스니펫에는 해당 필드 없음)
- user_id: 발급받은 유저와 동일 (로그인 사용자만)
- client_uuid: 사용 시점에 새로 생성 (다른 세션일 수 있으므로)

## 사전 조건
`generate_coupon_received_logs.ipynb`를 먼저 실행하여 `coupon_received_map.json`이 같은 디렉토리에 있어야 함

In [1]:
import random
import json
import uuid
from datetime import datetime, timedelta

In [2]:
# ───────────────────────────────────────────
# 설정값
# ───────────────────────────────────────────
END_DATE = datetime(2026, 6, 16, 23, 59, 59)

# 발급받은 쿠폰 중 실제 사용하는 비율
USE_RATE = 0.3

# 사용 시각 하한: 발급 후 최소 이만큼은 지나야 사용 (즉시 사용 방지)
USE_DELAY_MIN_SEC = 60

In [3]:
# coupon_received 결과 로드
with open('coupon_received_map.json', 'r', encoding='utf-8') as f:
    received_map = json.load(f)

print(f'✅ coupon_received_map.json 로드 완료 → 발급 기록 {len(received_map)}건')

✅ coupon_received_map.json 로드 완료 → 발급 기록 500건


In [4]:
def format_kst(dt):
    """event_timestamp 포맷 (KST +09:00)"""
    return dt.strftime('%Y-%m-%dT%H:%M:%S.') + f"{dt.microsecond // 1000:03d}+09:00"

def format_history_ts(dt):
    """history_timestamp 포맷 (마이크로초 포함)"""
    return dt.strftime('%Y-%m-%d %H:%M:%S.%f')

def parse_kst(ts_str):
    """'YYYY-MM-DDTHH:MM:SS.fff+09:00' -> datetime (naive, KST 기준)"""
    return datetime.strptime(ts_str[:-6], '%Y-%m-%dT%H:%M:%S.%f')

In [5]:
# 발급 기록 중 USE_RATE 만큼 랜덤 샘플링
use_count = int(len(received_map) * USE_RATE)
used_records = random.sample(received_map, use_count)

rows = []
skipped = 0

for record in used_records:
    user_id         = record['user_id']
    user_login_id   = record['user_login_id']
    coupon_code     = record['coupon_code']
    discount_amount = record['discount_amount']
    expired_days    = record['expired_days']
    received_ts     = parse_kst(record['received_event_ts'])

    # 사용 가능 구간: [발급 + USE_DELAY_MIN_SEC, 발급 + expired_days] (만료일 이내)
    earliest = received_ts + timedelta(seconds=USE_DELAY_MIN_SEC)
    latest   = received_ts + timedelta(days=expired_days)
    if latest > END_DATE:
        latest = END_DATE

    if earliest >= latest:
        # 사용 가능 구간이 없으면 스킵
        skipped += 1
        continue

    delta_sec = int((latest - earliest).total_seconds())
    event_ts  = earliest + timedelta(seconds=random.randint(0, delta_sec))

    history_ts = event_ts + timedelta(seconds=1)

    client_uuid = str(uuid.uuid4())

    json_log = json.dumps({
        'event_name':      'coupon_used',
        'couponCode':      coupon_code,
        'discountAmount':  discount_amount,
        'user_id':         user_id,
        'user_login_id':   user_login_id,
        'client_uuid':     client_uuid,
        'event_timestamp': format_kst(event_ts)
    }, ensure_ascii=False)

    rows.append((history_ts, json_log))

print(f'✅ {len(rows)}개 coupon_used 로그 생성 완료 (발급 {len(received_map)}건 중 샘플 {use_count}건, 사용가능구간 없어 스킵 {skipped}건)')

✅ 150개 coupon_used 로그 생성 완료 (발급 500건 중 샘플 150건, 사용가능구간 없어 스킵 0건)


In [6]:
# SQL 생성 및 저장
lines  = ['INSERT INTO first_save_history (history_timestamp, json_log) VALUES']
values = []

for history_ts, json_log in rows:
    ts_str  = format_history_ts(history_ts)
    escaped = json_log.replace("'", "''")
    values.append(f"  ('{ts_str}', '{escaped}')")

lines.append(',\n'.join(values) + ';')
sql = '\n'.join(lines)

with open('coupon_used_logs.sql', 'w', encoding='utf-8') as f:
    f.write(sql)

print(f'✅ {len(rows)}개 coupon_used 로그 SQL 생성 완료 → coupon_used_logs.sql')

✅ 150개 coupon_used 로그 SQL 생성 완료 → coupon_used_logs.sql


In [7]:
# ── 미리보기 ──
print('=== COUPON USED SQL (앞 500자) ===')
print(sql[:500])

=== COUPON USED SQL (앞 500자) ===
INSERT INTO first_save_history (history_timestamp, json_log) VALUES
  ('2026-02-26 01:59:05.000000', '{"event_name": "coupon_used", "couponCode": "REGULAR5000", "discountAmount": 5000, "user_id": 24, "user_login_id": "user0024", "client_uuid": "ca8e6eab-35fe-4a95-993f-9d3bb352119c", "event_timestamp": "2026-02-26T01:59:04.000+09:00"}'),
  ('2025-12-10 14:11:14.000000', '{"event_name": "coupon_used", "couponCode": "REGULAR5000", "discountAmount": 5000, "user_id": 86, "user_login_id": "user0086", 
